# PJM OASIS Java CLI live demo

Run this notebook top to bottom to test PJM OASIS access through the official `pjm-cli.jar` subprocess path. Fill in the configuration cell or set the matching environment variables before running the smoke test.

This notebook intentionally keeps TEST and STAGE URLs blank. TRAIN and PRODUCTION are the public OASIS environments; if you have private TEST or STAGE URLs, paste them into `ENV_URLS` locally.

In [ ]:
from __future__ import annotations

import getpass
import os
import subprocess
from collections.abc import Iterable
from pathlib import Path

MEM_ARGS = ["-Xms64m", "-Xmx256m"]
TIMEOUT_FLAG = ["-z", "180000"]
PROCESS_TIMEOUT_SEC = 120

ENV_URLS: dict[str, str] = {
    "TEST": "",
    "STAGE": "",
    "TRAIN": "https://oasisrefreshtrain.pjm.com/OASIS/",
    "PRODUCTION": "https://pjmoasis.pjm.com/OASIS/",
}

DEFAULT_ENV = "TRAIN"
READ_METHODS = {"GET", "HEAD", "OPTIONS"}
INPUT_TEMPLATES = {"PJMTRANSREQ", "PJMANNULMENT", "PJMTSRCOMMENT"}

# Set either here or through environment variables before running the notebook.
JAVA_PATH = os.getenv("PJM_CLI_JAVA_PATH", "java")
JAR_PATH = os.getenv("PJM_CLI_JAR_PATH", "")
USER = os.getenv("PJM_USERNAME", os.getenv("PJM_CLI_USER", ""))
PASSWORD = os.getenv("PJM_PASSWORD", os.getenv("PJM_CLI_PASSWORD", ""))
CERTIFICATE_PATH = os.getenv("PJM_CERT_PATH", "")
CERTIFICATE_PASSWORD = os.getenv("PJM_CERT_PASSWORD", "")
DOWNLOADS = Path(os.getenv("PJM_CLI_DOWNLOADS", "downloads")).expanduser()

# Production read requests are allowed after a warning. Writes are blocked unless this is True.
SHOW_PRODUCTION_WARNING = os.getenv("PJM_DISABLE_PRODUCTION_WARNING", "") != "1"
ALLOW_PRODUCTION_WRITE = os.getenv("PJM_ALLOW_PRODUCTION_WRITE", "") == "1"

print("default_env:", DEFAULT_ENV)
print("java:", JAVA_PATH)
print("jar:", JAR_PATH or "(not set)")
print("downloads:", DOWNLOADS)
print("test_url:", ENV_URLS["TEST"] or "(blank; set privately if needed)")
print("stage_url:", ENV_URLS["STAGE"] or "(blank; set privately if needed)")

In [ ]:
if not JAR_PATH:
    JAR_PATH = input("Path to pjm-cli.jar: ").strip()
if not USER:
    USER = input("PJM username: ").strip()
if not PASSWORD:
    PASSWORD = getpass.getpass("PJM password: ")
if not CERTIFICATE_PATH:
    CERTIFICATE_PATH = input("Path to login .p12/.pfx file: ").strip()
if not CERTIFICATE_PASSWORD:
    CERTIFICATE_PASSWORD = getpass.getpass("Certificate password: ")

jar_path = Path(JAR_PATH).expanduser()
cert_path = Path(CERTIFICATE_PATH).expanduser()

if not jar_path.exists():
    raise FileNotFoundError(f"pjm-cli.jar not found: {jar_path}")
if not cert_path.exists():
    raise FileNotFoundError(f"Certificate not found: {cert_path}")
if not USER or not PASSWORD:
    raise ValueError("PJM username and password are required")

DOWNLOADS.mkdir(parents=True, exist_ok=True)
CERTIFICATE = f"{cert_path}|{CERTIFICATE_PASSWORD}" if CERTIFICATE_PASSWORD else str(cert_path)

print("configuration ready")
print("jar:", jar_path)
print("cert:", cert_path)
print("downloads:", DOWNLOADS)

In [ ]:
def filename_only(value: str) -> str:
    return value.replace("\\", "/").split("/")[-1]


def env_url(env: str) -> str:
    key = env.upper()
    if key not in ENV_URLS:
        valid = ", ".join(sorted(ENV_URLS))
        raise ValueError(f"Unknown environment {env!r}. Valid: {valid}")
    url = ENV_URLS[key].strip()
    if not url:
        raise ValueError(f"No OASIS URL configured for {key}. Fill ENV_URLS[{key!r}] first.")
    return url if url.endswith("/") else f"{url}/"


def mask_command(cmd: Iterable[str]) -> list[str]:
    masked = list(cmd)
    for flag in ("-p", "-r"):
        if flag in masked:
            idx = masked.index(flag)
            if idx + 1 < len(masked):
                masked[idx + 1] = "***REDACTED***"
    return masked


def production_guard(env: str, *, method: str = "GET", template: str = "") -> None:
    if env.upper() != "PRODUCTION":
        return
    if SHOW_PRODUCTION_WARNING:
        print("WARNING: PRODUCTION environment selected. Read-only OASIS requests are allowed.")
    method_upper = method.upper()
    template_upper = template.upper()
    if ALLOW_PRODUCTION_WRITE:
        return
    if method_upper not in READ_METHODS or template_upper in INPUT_TEMPLATES:
        raise RuntimeError(
            "Blocked PRODUCTION write/reservation action. Use TRAIN for testing, "
            "or set ALLOW_PRODUCTION_WRITE=True only when intentionally writing to production."
        )


def build_cli_command(
    *,
    env: str,
    action: str,
    queries: Iterable[str],
    outfile: str,
    method: str = "GET",
    template: str = "",
) -> list[str]:
    production_guard(env, method=method, template=template)
    cmd = [
        JAVA_PATH,
        *MEM_ARGS,
        "-jar",
        str(jar_path),
        "-d",
        str(DOWNLOADS),
        "-u",
        USER,
        "-p",
        PASSWORD,
        "-r",
        CERTIFICATE,
        "-s",
        env_url(env),
        "-a",
        action,
    ]
    if method.upper() != "GET":
        cmd.extend(["-t", method.upper()])
    for query in queries:
        cmd.extend(["-q", query])
    cmd.extend(["-o", filename_only(outfile)])
    cmd.extend(TIMEOUT_FLAG)
    return cmd


def run_cli(
    cmd: list[str], *, timeout_sec: int = PROCESS_TIMEOUT_SEC
) -> subprocess.CompletedProcess[str]:
    print("command:")
    print(" ".join(mask_command(cmd)))
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec)
    print("returncode:", result.returncode)
    print("stdout:\n" + (result.stdout or "").strip())
    print("stderr:\n" + (result.stderr or "").strip())
    return result

In [ ]:
SMOKE_QUERIES = [
    "TEMPLATE=TRANSSERV",
    "OUTPUT_FORMAT=DATA",
    "PRIMARY_PROVIDER_CODE=PJM",
    "PRIMARY_PROVIDER_DUNS=073647877",
    "RETURN_TZ=EP",
    "VERSION=3.3",
]

smoke_cmd = build_cli_command(
    env=DEFAULT_ENV,
    action="/rest/secure/transserv",
    queries=SMOKE_QUERIES,
    outfile=f"smoke_{DEFAULT_ENV.lower()}.txt",
    template="TRANSSERV",
)
smoke_result = run_cli(smoke_cmd)
smoke_ok = (
    smoke_result.returncode == 0
    and "SUCCESS" in (smoke_result.stdout or "")
    and not (smoke_result.stderr or "").strip()
)
print(f"[SMOKE TEST:{DEFAULT_ENV}] {'PASS' if smoke_ok else 'FAIL'}")

In [ ]:
SMOKE_ENVIRONMENTS = ["TRAIN"]
# After privately filling ENV_URLS, you may use:
# SMOKE_ENVIRONMENTS = ["TEST", "STAGE", "TRAIN"]

for env in SMOKE_ENVIRONMENTS:
    print("=" * 72)
    print(f"Running smoke test for {env} ...")
    try:
        cmd = build_cli_command(
            env=env,
            action="/rest/secure/transserv",
            queries=SMOKE_QUERIES,
            outfile=f"smoke_{env.lower()}.txt",
            template="TRANSSERV",
        )
        result = run_cli(cmd)
        ok = (
            result.returncode == 0
            and "SUCCESS" in (result.stdout or "")
            and not (result.stderr or "").strip()
        )
        print(f"[SMOKE TEST:{env}] {'PASS' if ok else 'FAIL'}")
    except Exception as exc:
        print(f"Result [{env}]: ERROR - {exc}")

In [ ]:
RUN_PRODUCTION_READ_SMOKE = False

if RUN_PRODUCTION_READ_SMOKE:
    prod_cmd = build_cli_command(
        env="PRODUCTION",
        action="/rest/secure/transserv",
        queries=SMOKE_QUERIES,
        outfile="smoke_production.txt",
        template="TRANSSERV",
    )
    prod_result = run_cli(prod_cmd)
    prod_ok = (
        prod_result.returncode == 0
        and "SUCCESS" in (prod_result.stdout or "")
        and not (prod_result.stderr or "").strip()
    )
    print(f"[SMOKE TEST:PRODUCTION] {'PASS' if prod_ok else 'FAIL'}")
else:
    print("Production read smoke is disabled. Set RUN_PRODUCTION_READ_SMOKE=True to run it.")

In [ ]:
# This is a guard demo only. It does not submit a reservation or run pjm-cli.jar.
try:
    production_guard("PRODUCTION", method="PUT", template="pjmtransreq")
except RuntimeError as exc:
    print("Blocked as expected:", exc)

## What to do next

If the TRAIN smoke test passes, use the same command pattern for read-only template queries. Keep PRODUCTION writes blocked unless you are intentionally performing a production action and have set `ALLOW_PRODUCTION_WRITE=True` for that run.